In [ ]:
from datetime import datetime
from pathlib import Path

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("../../data/DieCasting_Quality_Raw_Data.csv", skiprows=1)
df["defect"] = df[df.columns[31:]].max(axis=1).astype(bool).astype(int)
df = df[df.columns[:31].tolist() + ["defect"]]
df.groupby("defect").size()

In [ ]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.lightgbm.autolog(disable=True)
experiment = mlflow.get_experiment_by_name("exp01")
run_name = f"02_consider_unbalanced_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
    source_file = globals().get("__vsc_ipynb_file__", "notebook.ipynb")
    # Split data into features (X) and target (y)
    X = df.drop(["id", "defect"], axis=1)
    y = df["defect"]

    # First split into train+val (80%) and test (20%)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Then split train+val into train (60%) and val (20%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.25, random_state=42
    )

    # Create dataset for LightGBM
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)

    # Set parameters for LightGBM
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.9,
        "is_unbalance": True,
    }

    # Train model
    num_round = 100
    lgb_model = lgb.train(
        params, train_data, num_round, valid_sets=[train_data, val_data]
    )

    val_predictions = (lgb_model.predict(X_val) > 0.5).astype(int)
    print("\nValidation Set Performance:")
    val_report = classification_report(y_val, val_predictions, output_dict=True)

    mlflow.log_metric("val_precision", val_report["1"]["precision"])
    mlflow.log_metric("val_recall", val_report["1"]["recall"])
    mlflow.log_metric("val_accuracy", val_report["accuracy"])


In [ ]:
import tempfile

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

with mlflow.start_run(run_id=run.info.run_id) as run:
    # Evaluate on validation set

    # Evaluate on test set
    test_predictions = (lgb_model.predict(X_test) > 0.5).astype(int)
    print("\nTest Set Performance:")
    test_report = classification_report(y_test, test_predictions, output_dict=True)
    mlflow.log_metric("test_precision", test_report["1"]["precision"])
    mlflow.log_metric("test_recall", test_report["1"]["recall"])
    mlflow.log_metric("test_accuracy", test_report["accuracy"])

    disp = ConfusionMatrixDisplay.from_predictions(y_test, test_predictions)
    with tempfile.TemporaryDirectory() as temp_dir:
        path = Path(temp_dir) / "confusion_matrix.png"
        plt.savefig(path)
        mlflow.log_artifact(local_path=path)

    mlflow.lightgbm.log_model(
        lgb_model,
        artifact_path="model",
        signature=mlflow.models.infer_signature(X_test, test_predictions),
        input_example=X_test.iloc[:5],
    )